In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import bagpipes as pipes
from sparcl.client import SparclClient

In [ ]:
import arya
arya.init("mnras")

In [ ]:
import arya
arya.init("mnras")

In [ ]:
plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "sans"

In [ ]:
client = SparclClient()

In [ ]:
def load_full_spec(targetid):
    inc = ['sparcl_id', 'specid', 'data_release', 'redshift', 'flux',
       'wavelength', 'model', 'ivar', 'mask', 'spectype', 'ra', 'dec', 'wave_sigma']


    galspec_raw = client.retrieve_by_specid(specid_list = [int(targetid)], 
        include = inc,
        dataset_list = ['DESI-DR1'], 
        limit=1_000_000,
        )[1]
    return galspec_raw

In [ ]:
def load_spectrum(targetid):
    inc = ['sparcl_id', 'specid', 'data_release', 'redshift', 'flux',
       'wavelength', 'model', 'ivar', 'mask', 'spectype', 'ra', 'dec', 'wave_sigma']


    galspec_raw = client.retrieve_by_specid(specid_list = [int(targetid)], 
        include = inc,
        dataset_list = ['DESI-DR1'], 
        limit=1_000_000,
        )[1]

    return np.c_[
        galspec_raw["wavelength"], 
        galspec_raw["flux"] * 1e-17,
        1 / np.sqrt(galspec_raw["ivar"]) * 1e-17,
    ]


In [ ]:
target_id = 39627711919951479

In [ ]:
galaxy = pipes.galaxy(target_id, load_spectrum, photometry_exists=False)

In [ ]:
spec = load_full_spec(target_id)

In [ ]:
galaxy

In [ ]:
galaxy.plot()

In [ ]:
dblplaw = {}                        
dblplaw["tau"] = (0., 15.)            
dblplaw["alpha"] = (0.01, 1000.)
dblplaw["beta"] = (0.01, 1000.)
dblplaw["alpha_prior"] = "log_10"
dblplaw["beta_prior"] = "log_10"
dblplaw["massformed"] = (1., 15.)
dblplaw["metallicity"] = (0.1, 2.)
dblplaw["metallicity_prior"] = "log_10"

nebular = {}
nebular["logU"] = (-4, -1.)

dust = {}
dust["type"] = "CF00"
dust["eta"] = 2.
dust["Av"] = (0., 2.0)
dust["n"] = (0.3, 2.5)
dust["n_prior"] = "Gaussian"
dust["n_prior_mu"] = 0.7
dust["n_prior_sigma"] = 0.3



In [ ]:


continuity = {}
continuity["massformed"] = (0., 13.)
continuity["metallicity"] = (0.01, 5.)
continuity["metallicity_prior"] = "log_10"
continuity["bin_edges"] = [0., 10., 100., 250., 500.,
                           1000., 3000., 10_000.]

for i in range(1, len(continuity["bin_edges"])-1):
    continuity["dsfr" + str(i)] = (-10., 10.)
    continuity["dsfr" + str(i) + "_prior"] = "student_t"
    #continuity["dsfr" + str(i) + "_prior_scale"] = 0.3  # Defaults to this value as in Leja19, but can be set
    #continuity["dsfr" + str(i) + "_prior_df"] = 2       # Defaults to this value as in Leja19, but can be set



In [ ]:
noise = {}
noise["type"] = "white_scaled"
noise["scaling"] = (1., 10.)
noise["scaling_prior"] = "log_10"


In [ ]:
fit_instructions = {}
fit_instructions["redshift"] = (0.0, 0.4)
fit_instructions["t_bc"] = 0.01
fit_instructions["redshift_prior"] = "Gaussian"
fit_instructions["redshift_prior_mu"] = spec["redshift"]
fit_instructions["redshift_prior_sigma"] = 0.05
# fit_instructions["dblplaw"] = dblplaw 
fit_instructions["nebular"] = nebular
fit_instructions["dust"] = dust

fit_instructions["noise"] = noise


fit_instructions["continuity"] = continuity


In [ ]:
fit = pipes.fit(galaxy, fit_instructions, run="spec_continuity_u")

fit.fit(verbose=True)

In [ ]:
fit.posterior

In [ ]:
%matplotlib inline

In [ ]:
from desi_utils.plotting import plot_wrapped_spectrum

In [ ]:
fig, axs = fit.plot_spectrum_posterior(save=False, show=False)
ax[0].set(
    yscale = "asinh"
)

plt.ylim(1e-3,  1.5)
plt.yscale("log")


In [ ]:
model_spec = fit.posterior.fitted_model.model_galaxy.spectrum

In [ ]:
scale = 1e-17
spectra = {
    "galaxy": (galaxy.spectrum[:, 0], galaxy.spectrum[:, 1] / scale),
    "posterior": (model_spec[:, 0], model_spec[:, 1] / scale),
}

In [ ]:
plot_wrapped_spectrum(spectra, yscale="log", ylim=(0.1, 1e3))

In [ ]:
fig = fit.plot_sfh_posterior(save=False, show=True)

In [ ]:
fig = fit.plot_corner(save=False, show=True)